In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.bfloat16

In [2]:
from stable_audio_vae import StableAudioVAE

vae = StableAudioVAE(
    config_path="./models/ace-step-vae/config.json",
    checkpoint_path="./models/ace-step-vae/checkpoint.ckpt",
).to(device).to(model_dtype)

C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [3]:
from transformers import AutoModel

model = AutoModel.from_pretrained("ACE-Step/acestep-v15-turbo-shift1", trust_remote_code=True,
                                  attn_implementation="eager", dtype=model_dtype).to(device).to(model_dtype)

C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:454: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:639: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\finite_scalar_quantization.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\lookup_free_quantization.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use 

In [4]:
def cast_input_hook(module, args):
    return tuple(a.to(module.weight.dtype) if isinstance(a, torch.Tensor) else a for a in args)

model.tokenizer.quantizer.project_out.register_forward_pre_hook(cast_input_hook)

In [30]:
def save_audio(audio_data):

    from pathlib import Path
    import soundfile as sf

    output_path = Path("./outputs/output.wav")
    audio_np = audio_data.to(torch.float32).cpu().detach().numpy().T.reshape((-1, 2))
    sf.write(output_path, audio_np, 48000)
    return audio_np

In [69]:
text_hidden_states = torch.zeros(2, 77, 1024, dtype=model_dtype, device=device)
text_attention_mask = torch.zeros(2, 77, dtype=model_dtype, device=device)
lyric_hidden_states = torch.zeros(2, 123, 1024, dtype=model_dtype, device=device)
lyric_attention_mask = torch.zeros(2, 123, dtype=model_dtype, device=device)
refer_audio_acoustic_hidden_states_packed = torch.zeros(3, 750, 64, dtype=model_dtype, device=device)
refer_audio_order_mask = torch.LongTensor([0, 0, 1]).to(device)


is_covers = torch.Tensor([False])

seconds = 20
infer_steps = 250
frames_per_second = 25

seq_len = int(seconds * frames_per_second / 2)

cur_chunk_mask = torch.ones(2, seq_len, 64, dtype=model_dtype, device=device)
cur_silence_latent = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)
cur_src_latents = torch.randn(2, seq_len, 64, dtype=model_dtype, device=device)

print(cur_chunk_mask.shape)
print(cur_silence_latent.shape)
print(cur_src_latents.shape)

torch.Size([2, 250, 64])
torch.Size([2, 250, 64])
torch.Size([2, 250, 64])


In [70]:
outputs = model.generate_audio(
    text_hidden_states=text_hidden_states,
    text_attention_mask=text_attention_mask,
    lyric_hidden_states=lyric_hidden_states,
    lyric_attention_mask=lyric_attention_mask,
    refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
    refer_audio_order_mask=refer_audio_order_mask,
    src_latents=cur_src_latents,
    chunk_masks=cur_chunk_mask,
    silence_latent=cur_silence_latent,
    infer_steps=infer_steps,
    is_covers=is_covers,
)

In [71]:
output_latents = outputs['target_latents'].transpose(1, 2).contiguous()
output = vae.decode(output_latents)
print(output.shape)
audio_np = save_audio(output)
print(audio_np.shape)

torch.Size([2, 2, 480000])
(960000, 2)
